> 在线讲义：[第 59 章 · 数据输入、checkpoint 与故障恢复：几周的作业怎么扛住中断](https://wikieden.github.io/ai-infra-from-constraints/lessons/59-data-checkpoint.html) · 本书《从约束推导 AI Infra》

# 第 59 章 · 数据输入、checkpoint 与故障恢复：几周的作业怎么扛住中断

配套讲义：仓库里 `ai-infra/notes/lessons/59-data-checkpoint.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



数据准备进程数、checkpoint 内容与重分片、异步保存的可用恢复点、最优保存间隔、掉队者与损失尖峰，取自 AII 第 10 章 §10.4。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
N = 8_190_735_360
CKPT = 14 * N                    # 权重 2 + 主权重 4 + 两份矩 8 字节/参数
MTBF_CARD = 1024 * 7.9 * 3600    # 1024 卡 7.9 h 中断一次 → 每卡约 337 天

**函数 `workers_needed`**：小工具。

In [ ]:
def workers_needed(tok_s: float, per_worker_seq: float = 2, seq: int = 8192) -> int:
    return math.ceil(tok_s / seq / per_worker_seq)

**函数 `commit_times`**：异步保存：先复制 copy 秒到缓冲，再按 bw 上传；上一份传完才传下一份。

In [ ]:
def commit_times(capture: list[float], size: float, bw: float, copy: float = 0.5) -> list[float]:
    """异步保存：先复制 copy 秒到缓冲，再按 bw 上传；上一份传完才传下一份。"""
    out, free = [], 0.0
    for t in capture:
        start = max(t + copy, free)
        free = start + size / bw
        out.append(free)
    return out

**函数 `overhead`**：L(τ) = c/τ + λτ/2 + λr 的三项。

In [ ]:
def overhead(tau: float, c: float, lam: float, r: float = 120) -> tuple[float, float, float]:
    """L(τ) = c/τ + λτ/2 + λr 的三项。"""
    return c / tau, lam * tau / 2, lam * r

**函数 `tau_opt`**：小工具。

In [ ]:
def tau_opt(c: float, lam: float) -> float:
    return math.sqrt(2 * c / lam)

**函数 `job_rate`**：小工具。

In [ ]:
def job_rate(cards: int) -> float:
    return cards / MTBF_CARD

**函数 `expected_max_std`**：n 个独立标准正态的最大值的期望（数值积分）。

In [ ]:
def expected_max_std(n: int) -> float:
    """n 个独立标准正态的最大值的期望（数值积分）。"""
    if n == 1:
        return 0.0
    phi = lambda x: math.exp(-x * x / 2) / math.sqrt(2 * math.pi)
    Phi = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))
    h, s = 0.001, 0.0
    x = -8.0
    while x < 8:
        s += x * n * phi(x) * Phi(x) ** (n - 1) * h
        x += h
    return s

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：数据输入")
need = 1e11 / (25 * 86400)
print(f"  {need:,.0f} token/s = {need / 8192:.1f} 条/s，int64 只有 {need * 8 / 1e6:.2f} MB/s；每进程 2 条/s → 至少 {workers_needed(need)} 个")
print(f"  48 卡每步 52.3 s 处理 384 条 → {384 / 52.3:.1f} 条/s；四个进程 8 条/s 够用")
print("  训练完成 batch 100、预取到 108：恢复从 101 开始，不是 109")

In [ ]:
print("\n第二步：checkpoint 内容")
print(f"  每参数 14 B → {CKPT / 1e9:.1f} GB；门控投影 96 MiB × (1 + 2 + 2 + 2) = {96 * 7} MiB；TP4 → TP8 每片 24 → 12 MiB")

In [ ]:
print("\n第三步：异步保存与可用恢复点（112 GB）")
for bw in (7e9, 20e9):
    ts = commit_times([20, 40], 112e9, bw)
    ok = [t for t, cm in zip((20, 40), ts) if cm <= 50]
    print(f"  {bw / 1e9:.0f} GB/s：提交于 {ts[0]:.1f} / {ts[1]:.1f} s；50 s 故障恢复到 {max(ok)} s，重做 {50 - max(ok)} s")
print(f"  每 10 s 一份要 {112 / 10:.1f} GB/s > 7 → 队列增长；每 20 s 一份 {112 / 20:.1f} GB/s")

In [ ]:
print("\n第四步：保存间隔")
c = CKPT / 7e9
lam = job_rate(1024)
print(f"  1024 卡 λ = 1/{1 / lam / 3600:.1f} h，c = {c:.1f} s，τ* = {tau_opt(c, lam):.0f} s")
for tau in (300, 900, 1800):
    a, b, r = overhead(tau, c, lam)
    print(f"  τ = {tau:>4} s：保存 {a * 100:.1f}% + 重做 {b * 100:.1f}% + 恢复 {r * 100:.1f}% = {(a + b + r) * 100:.1f}%")
for cards in (32, 48):
    print(f"  {cards} 卡 τ = 1800 s：{sum(overhead(1800, c, job_rate(cards))) * 100:.2f}%")

In [ ]:
print("\n第五步：掉队者（μ = 52.2 s，另加 0.58 s）")
for n in (8, 48, 1024):
    k = expected_max_std(n)
    print(f"  {n:>4} 卡：E[max] − μ = {k:.2f}σ；σ = 2% → {52.2 + k * 1.044 + 0.58:.1f} s，5% → {52.2 + k * 2.61 + 0.58:.1f} s")
for n in (8, 48):
    wait = max(0, 3 - expected_max_std(n - 1))
    re = 52.2 * n / (n - 1) + expected_max_std(n - 1) * 1.044 + 0.58
    print(f"  {n} 卡一张慢到 μ + 3σ：等待多 {wait:.2f}σ；重分配每步 {re:.1f} s")
print(f"  驱逐：τ/2 + r = {600 / 2 + 120} s ≈ {420 / 52.8:.0f} 步")
l48, ls = job_rate(48), 1 / (7 * 86400)
print(f"  损失尖峰每 7 天一次：τ* {tau_opt(c, l48):.0f} → {tau_opt(c, l48 + ls):.0f} s；τ = 600 s {sum(overhead(600, c, l48)) * 100:.2f}% → {(c / 600 + (l48 + ls) * 420) * 100:.2f}%；1800 s {sum(overhead(1800, c, l48)) * 100:.2f}% → {(c / 1800 + (l48 + ls) * 1020) * 100:.2f}%")

## 3. 代码里才看得到的

1. checkpoint 用 14N = 114.7 GB，保存 16.38 s；用 115 GB / 16.4 s 算出的 τ* 会差几秒（4465 vs 4458）。
2. `commit_times` 让第二份必须等第一份传完才开始上传，所以 7 GB/s 时第二份是 56.5 s，不是 40.5 + 16。
3. `expected_max_std` 用数值积分 $\int x \cdot N\varphi(x)\Phi(x)^{N-1}dx$，步长 0.001 已足够到小数点后两位。

## 4. 自己改一改

- 给 `overhead` 加"异步保存只停 0.5 s，但恢复点平均晚 c 秒"的版本。
- 把 `commit_times` 改成可以并行上传两份，看提交时刻怎么变。
- 用 `expected_max_std` 画 k_N 随 N 的曲线，和 $\sqrt{2\ln N}$ 比。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 送的是小盒子，备菜要人手；读档从训练到的地方，不从备好的地方；
> 存档要存背包、进度和种子；
> 交给快递员就能走，签收前丢了还算上一个；
> 存盘太勤耽误工夫，太少死机丢一截，根号二 c 除以 λ；
> 全班等最后一个，偶尔慢就等，一直慢才请出去。